##**Aviso:** Antes de começar, vá no menu superior do Google Colab em Ambiente de execução > Alterar o tipo de ambiente de execução e escolha T4 GPU. Se você não fizer isso, o modelo vai rodar usando a CPU e a resposta demorará minutos em vez de segundos.

In [1]:
# Instalação do zstd (dependência) e depois do Ollama
!apt-get update && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 https://cli.github.com/packages stable InRelease [3,917 B]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,600 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/universe am

```markdown
### Próximos Passos
Após a instalação, é necessário iniciar o servidor do Ollama em segundo plano para começar a baixar e rodar modelos.
```

In [2]:
# Iniciar o servidor em background
import subprocess
import os

# Rodando o servidor
subprocess.Popen(['ollama', 'serve'], stdout=open(os.devnull, 'w'), stderr=open(os.devnull, 'w'))

<Popen: returncode: None args: ['ollama', 'serve']>

## Executando o ollama

Após iniciar a execução do servidor em segundo plano iremos instalar o modelo `gemma4`.

Optou-se pelo gemma4 pois ele é a família de modelos abertos do Google e se destaca principalmente por rodar totalmente offline e local em celulares e computadores.

Ele não serve só para conversar. **O Gemma 4 possui recursos para criar "agentes" autônomos**, capazes de tomar decisões, ler e executar etapas para finalizar tarefas por conta própria (como enviar e-mails, ler dados e manipular arquivos).

**Observação:** a instalação do modelo pode demorar alguns minutos

In [3]:
!ollama pull gemma4

In [4]:
# Lista todos os modelos baixados no Ollama
!ollama list

NAME             ID              SIZE      MODIFIED               
gemma4:latest    c6eb396dbd59    9.6 GB    Less than a second ago    


## Instalando o streamlit e o localtunnel

Como o código está sendo executado em um Jupyter Notebook no Google Colab, será preciso contornar o isolamento de rede do Google, pois não é possível simplesmente abrir o localhost do Streamlit aqui.

Será utilizado o Streamlit combinado com o Localtunnel no Google Colab para criar interfaces gráficas interativas e acessá-las pelo navegador usando o hardware gratuito do Google.

Como o Google Colab roda em servidores virtuais na nuvem, ele não possui uma tela própria para exibir sites ou aplicativos tradicionais e bloqueia conexões externas diretas. Essa dupla resolve exatamente esse problema sem precisar gastar nada ou configurar ferramentas complexas.

In [5]:
# 1. Instalação do streamlit
!pip install streamlit pandas requests -q

# 2. Instalação do Localtunnel (para gerar o link público)
!npm install localtunnel -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 119.1 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
added 22 packages in 2s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇

## Populando a pasta `data/` com os dados necessários

Para não precisar fazer upload manual de nenhum arquivo, este bloco de código Python cria a pasta `data/` e gera automaticamente todos os JSONs e CSVs do PoupaSonho.

Observe que essa ação pode ser realizada manualmente, criando a pasta e em seguida inserindo os arquivos dentro da pasta.

In [6]:
import os
import json
import pandas as pd

# Cria a pasta data
os.makedirs('./data', exist_ok=True)

# 1. Cria perfil_investidor.json
perfil = {
  "nome": "João Silva",
  "idade": 32,
  "profissao": "Analista de Sistemas",
  "renda_mensal": 5000.00,
  "perfil_investidor": "moderado",
  "objetivo_principal": "Construir reserva de emergência",
  "patrimonio_total": 15000.00,
  "reserva_emergencia_atual": 10000.00,
  "aceita_risco": False,
  "metas": [
    {"meta": "Completar reserva de emergência", "valor_necessario": 15000.00, "prazo": "2026-06"},
    {"meta": "Entrada do apartamento", "valor_necessario": 50000.00, "prazo": "2027-12"}
  ]
}
with open('./data/perfil_investidor.json', 'w', encoding='utf-8') as f:
    json.dump(perfil, f, ensure_ascii=False, indent=2)

# 2. Cria desafios_economia.json
desafios = [
  {
    "id": "DES001",
    "titulo": "Semana Zero Delivery",
    "categoria": "alimentacao",
    "economia_estimada_semanal": 120.00,
    "descricao": "Substituir refeições de aplicativos por jantares cozinhados em casa."
  },
  {
    "id": "DES002",
    "titulo": "Corte de Streaming Fantasma",
    "categoria": "lazer",
    "economia_estimada_mensal": 55.90,
    "descricao": "Suspender temporariamente assinaturas de vídeo ou música que não sejam vitais este mês."
  },
  {
    "id": "DES003",
    "titulo": "Fim de Semana Offline de Compras",
    "categoria": "lazer",
    "economia_estimada_semanal": 100.00,
    "descricao": "Fazer apenas passeios gratuitos e evitar abrir aplicativos de e-commerce no sábado e domingo."
  }
]
with open('./data/desafios_economia.json', 'w', encoding='utf-8') as f:
    json.dump(desafios, f, ensure_ascii=False, indent=2)

# 3. Cria produtos_financeiros.json
produtos = [
  {"nome": "Tesouro Selic", "categoria": "renda_fixa", "risco": "baixo", "rentabilidade": "100% da Selic", "aporte_minimo": 30.00},
  {"nome": "CDB Liquidez Diária", "categoria": "renda_fixa", "risco": "baixo", "rentabilidade": "102% do CDI", "aporte_minimo": 100.00},
  {"nome": "Fundo de Ações", "categoria": "fundo", "risco": "alto", "rentabilidade": "Variável", "aporte_minimo": 100.00}
]
with open('./data/produtos_financeiros.json', 'w', encoding='utf-8') as f:
    json.dump(produtos, f, ensure_ascii=False, indent=2)

# 4. Cria transacoes.csv
transacoes_data = {
    'data': ['2025-10-01', '2025-10-02', '2025-10-03', '2025-10-05', '2025-10-07', '2025-10-10', '2025-10-12', '2025-10-15', '2025-10-20', '2025-10-25'],
    'descricao': ['Salário', 'Aluguel', 'Supermercado', 'Netflix', 'Farmácia', 'Restaurante', 'Uber', 'Conta de Luz', 'Academia', 'Combustível'],
    'categoria': ['receita', 'moradia', 'alimentacao', 'lazer', 'saude', 'alimentacao', 'transporte', 'moradia', 'saude', 'transporte'],
    'valor': [5000.0, 1200.0, 450.0, 55.9, 89.0, 120.0, 45.0, 180.0, 99.0, 250.0],
    'tipo': ['entrada', 'saida', 'saida', 'saida', 'saida', 'saida', 'saida', 'saida', 'saida', 'saida']
}
df_transacoes = pd.DataFrame(transacoes_data)
df_transacoes.to_csv('./data/transacoes.csv', index=False)

# 5. Cria historico_atendimento.csv
historico_data = {
    'data': ['2025-09-15', '2025-09-22', '2025-10-01', '2025-10-12', '2025-10-25'],
    'canal': ['chat', 'telefone', 'chat', 'chat', 'email'],
    'tema': ['CDB', 'Problema no app', 'Tesouro Selic', 'Metas financeiras', 'Atualização cadastral'],
    'resumo': [
        'Cliente perguntou sobre rentabilidade e prazos',
        'Erro ao visualizar extrato foi corrigido',
        'Cliente pediu explicação sobre o funcionamento do Tesouro Direto',
        'Cliente acompanhou o progresso da reserva de emergência',
        'Cliente atualizou e-mail e telefone'
    ],
    'resolvido': ['sim', 'sim', 'sim', 'sim', 'sim']
}
df_historico = pd.DataFrame(historico_data)
df_historico.to_csv('./data/historico_atendimento.csv', index=False)

print("Arquivos de dados (incluindo o novo histórico de atendimento) criados com sucesso na pasta /data!")

Arquivos de dados (incluindo o novo histórico de atendimento) criados com sucesso na pasta /data!


# Criando o `app.py`

O comando `%%writefile`  do Colab vai pegar todo o código do nosso Streamlit e salvar em um arquivo chamado app.py.

In [7]:
%%writefile app.py
import json
import pandas as pd
import requests
import streamlit as st
import os

OLLAMA_URL = "http://localhost:11434/api/generate"
MODELO = "gemma4"

def carregar_dados():
    perfil = json.load(open('./data/perfil_investidor.json', encoding="utf-8"))
    transacoes = pd.read_csv('./data/transacoes.csv')
    historico_atendimento = pd.read_csv('./data/historico_atendimento.csv')

    total_receitas = transacoes[transacoes['tipo'] == 'entrada']['valor'].sum()
    total_despesas = transacoes[transacoes['tipo'] == 'saida']['valor'].sum()
    sobra_real = total_receitas - total_despesas
    gastos_categoria = transacoes[transacoes['tipo'] == 'saida'].groupby('categoria')['valor'].sum().to_dict()

    try:
        desafios = json.load(open('./data/desafios_economia.json', encoding="utf-8"))
    except FileNotFoundError:
        desafios = []

    produtos_todos = json.load(open('./data/produtos_financeiros.json', encoding="utf-8"))
    produtos_seguros = [p for p in produtos_todos if p.get("risco") == "baixo"]

    return perfil, total_receitas, total_despesas, sobra_real, gastos_categoria, desafios, produtos_seguros, historico_atendimento

perfil, total_receitas, total_despesas, sobra_real, gastos_categoria, desafios, produtos_seguros, historico_atendimento = carregar_dados()

contexto = f"""
[DADOS DO CADASTRO DO CLIENTE]
- Nome: {perfil['nome']}
- Idade: {perfil['idade']} anos
- Renda Mensal Estável: R$ {total_receitas:.2f}
- Perfil de Risco: {perfil.get('perfil_investidor', 'Moderado').capitalize()}

[METAS ATIVAS NO SISTEMA]
{json.dumps(perfil.get('metas', []), indent=2, ensure_ascii=False)}

[DIAGNÓSTICO DA SAÚDE FINANCEIRA REAL]
- Total de Entradas: R$ {total_receitas:.2f}
- Total de Saídas: R$ {total_despesas:.2f}
- Margem de Sobra Livre: R$ {sobra_real:.2f}
- Gastos por Categoria: {json.dumps(gastos_categoria, indent=2, ensure_ascii=False)}

[HISTÓRICO DE ATENDIMENTOS ANTERIORES]
{historico_atendimento.to_string(index=False)}

[CATÁLOGO DE DESAFIOS DISPONÍVEIS]
{json.dumps(desafios, indent=2, ensure_ascii=False)}

[PRODUTOS SEGUROS (BAIXO RISCO)]
{json.dumps(produtos_seguros, indent=2, ensure_ascii=False)}
"""

SYSTEM_PROMPT = """[ROLE & IDENTITY]
- Nome: PoupaSonho
- Perfil: Assistente financeiro virtual focado no planejamento de micro-metas e economia.
- Tom de Voz: Motivador, acolhedor e sem julgamentos.

[CONTEXTUAL ANCHORING]
Trate o [CONTEXTO DO SISTEMA] como verdade absoluta. Considere o [HISTÓRICO DE ATENDIMENTOS ANTERIORES] para não repetir orientações já dadas e dar continuidade ao suporte.

[THOUGHT PROCESS]
1. Identifique o tema.
2. Verifique os números no contexto.
3. Se for meta, calcule a parcela mensal.
4. Cruze a categoria de maior gasto do usuário com um desafio.
5. Indique um produto seguro para guardar o dinheiro.

[REGRAS]
- NUNCA invente matemática ou taxas de juros.
- Recomende APENAS opções de baixo risco do contexto.
- Se a parcela da meta ultrapassar a margem de sobra real, sugira amigavelmente estender o prazo.
"""

def perguntar(msg, historico_chat):
    prompt_completo = f"{SYSTEM_PROMPT}\n\n[CONTEXTO DO SISTEMA]\n{contexto}\n\n[HISTÓRICO DA CONVERSA]\n{historico_chat}\n\nUsuário: {msg}\nPoupaSonho:"
    payload = {"model": MODELO, "prompt": prompt_completo, "stream": False}

    try:
        r = requests.post(OLLAMA_URL, json=payload)
        r.raise_for_status()
        return r.json()['response']
    except Exception as e:
        return f"⚠️ Erro de conexão com o Ollama: {e}"

st.set_page_config(page_title="PoupaSonho", page_icon="🐷")
st.title("🐷 PoupaSonho")
st.markdown("Seu assistente financeiro focado em transformar metas em realidade!")

if "mensagens" not in st.session_state:
    st.session_state.mensagens = []

for msg in st.session_state.mensagens:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

if pergunta := st.chat_input("Como posso te ajudar a bater suas metas hoje?"):
    st.session_state.mensagens.append({"role": "user", "content": pergunta})
    with st.chat_message("user"):
        st.markdown(pergunta)

    historico_texto = "\n".join([f"{m['role'].capitalize()}: {m['content']}" for m in st.session_state.mensagens[-4:]])

    with st.spinner("Calculando possibilidades..."):
        resposta = perguntar(pergunta, historico_texto)

    st.session_state.mensagens.append({"role": "assistant", "content": resposta})
    with st.chat_message("assistant"):
        st.markdown(resposta)

Writing app.py


#Rodando a Aplicação e Gerando o Link

Esta última célula vai revelar o endereço IP da máquina do Google e depois iniciar o aplicativo.

In [12]:
# devido à instabilidade do colab é recomendado iniciar novamente o ollama para garantir que ele está sendo executado em segundo plano antes de criar o link
subprocess.Popen(['ollama', 'serve'], stdout=open(os.devnull, 'w'), stderr=open(os.devnull, 'w'))

# 1. Descobrir o IP externo da máquina
print("=== COPIE O IP ABAIXO ===")
!wget -q -O - ipv4.icanhazip.com
print("=========================\n")

# 2. Rodar o Streamlit em background
!streamlit run app.py &>/content/logs.txt &

# 3. Gerar o link público usando localtunnel
!npx localtunnel --port 8501

=== COPIE O IP ABAIXO ===
34.91.117.235

⠙⠹your url is: https://two-cooks-ring.loca.lt
^C


Se aparecer algum erro quando a página do streamlite abrir, atualize a página algumas vezes até o erro desaparecer.